# Notebook 02 — YOLO Object Detection Demo: IOU + NMS From Scratch

Companion to `02-object-detection-with-yolo.md`. This notebook is **pure numpy** — no
trained model, no GPU, no external data. It implements the two pieces of YOLO's inference
pipeline that show up constantly in interviews: **Intersection over Union (IOU)** and
**Non-Max Suppression (NMS)** — applied to a handful of synthetic bounding boxes standing in
for raw YOLO detections around superscript-candidate regions.

`pip install numpy` is the only dependency.


## How a YOLO grid-cell prediction maps to these boxes

Before writing the boxes, a quick picture of *where* raw detections like these come from.
YOLO divides the input (here, imagine a cropped text-line image, per Chapter 05's
ROI-cropping discussion) into an `S x S` grid. Each cell predicts `B` boxes (one per anchor)
as offsets from that cell's anchor shapes, plus an objectness/confidence score:

```
Cropped text-line image, S=8 grid cells across:

  +----+----+----+----+----+----+----+----+
  |    |    |    |    |    |    |    |    |
  +----+----+----+----+----+----+----+----+
  |    |    |    |####|    |    |    |    |   <- cell (3, 1): true superscript '1' sits
  +----+----+----+----+----+----+----+----+      here; this cell (and its neighbors,
  |    |    |    | .. |    |    |    |    |      because the glyph straddles a boundary)
  +----+----+----+----+----+----+----+----+      each predict a box for it -- hence
  |    |    |    |    |    |    |####|    |      multiple overlapping raw detections.
  +----+----+----+----+----+----+----+----+

  Cell (6, 3) similarly fires on a second, separate true superscript elsewhere on the line.
```

Because nearby cells (and multiple anchors within one cell) commonly all fire with some
confidence around the same real object, a raw YOLO forward pass over-produces boxes per true
object — that's the `raw_boxes` array below, and it's exactly what NMS exists to clean up.


In [1]:
import numpy as np

# Synthetic raw YOLO-style detections for one cropped text-line image: [x1, y1, x2, y2, score]
# Simulates: 3 overlapping/duplicate boxes around ONE true superscript (candidate A),
# 2 overlapping/duplicate boxes around a SECOND, separate true superscript (candidate B),
# and 1 lower-confidence stray false-positive detection elsewhere (candidate C).
raw_boxes = np.array([
    [100, 40, 112, 54, 0.91],   # candidate A -- strongest detection
    [101, 41, 113, 55, 0.78],   # candidate A duplicate (should be suppressed)
    [99,  39, 111, 53, 0.65],   # candidate A duplicate (should be suppressed)
    [200, 60, 216, 76, 0.83],   # candidate B -- separate true superscript
    [201, 61, 215, 75, 0.55],   # candidate B duplicate (should be suppressed)
    [300, 45, 309, 56, 0.31],   # candidate C -- weak, isolated false-positive-ish detection
], dtype=np.float64)

print(f"{len(raw_boxes)} raw candidate boxes before NMS:")
print(raw_boxes)


6 raw candidate boxes before NMS:
[[100.    40.   112.    54.     0.91]
 [101.    41.   113.    55.     0.78]
 [ 99.    39.   111.    53.     0.65]
 [200.    60.   216.    76.     0.83]
 [201.    61.   215.    75.     0.55]
 [300.    45.   309.    56.     0.31]]


## Intersection over Union (IOU), from scratch

`IOU = area(A ∩ B) / area(A ∪ B)`. Implemented directly against the box coordinates, with no
dependency beyond numpy.


In [2]:
def iou(box_a, box_b):
    """IOU between two boxes in [x1, y1, x2, y2] format."""
    xa1, ya1, xa2, ya2 = box_a
    xb1, yb1, xb2, yb2 = box_b

    inter_x1 = max(xa1, xb1)
    inter_y1 = max(ya1, yb1)
    inter_x2 = min(xa2, xb2)
    inter_y2 = min(ya2, yb2)

    inter_w = max(0.0, inter_x2 - inter_x1)
    inter_h = max(0.0, inter_y2 - inter_y1)
    inter_area = inter_w * inter_h

    area_a = (xa2 - xa1) * (ya2 - ya1)
    area_b = (xb2 - xb1) * (yb2 - yb1)
    union_area = area_a + area_b - inter_area

    return inter_area / union_area if union_area > 0 else 0.0


# Sanity checks
print("IOU(candidate A #1, candidate A #2) =", round(iou(raw_boxes[0, :4], raw_boxes[1, :4]), 3))
print("IOU(candidate A #1, candidate B #1) =", round(iou(raw_boxes[0, :4], raw_boxes[3, :4]), 3))
print("IOU(identical box, itself)          =", iou(raw_boxes[0, :4], raw_boxes[0, :4]))


IOU(candidate A #1, candidate A #2) = 0.741
IOU(candidate A #1, candidate B #1) = 0.0
IOU(identical box, itself)          = 1.0


## Non-Max Suppression (NMS), from scratch

Sort by confidence descending; repeatedly keep the top box and drop every remaining box that
overlaps it above the IOU threshold (a near-duplicate detection of the same object); repeat
on what's left.


In [3]:
def non_max_suppression(boxes, iou_threshold=0.4):
    """boxes: numpy array of shape (N, 5) -> [x1, y1, x2, y2, score].
    Returns the kept boxes as a numpy array, highest-confidence survivor of each cluster first.
    """
    if len(boxes) == 0:
        return np.empty((0, 5))

    order = boxes[:, 4].argsort()[::-1]  # indices sorted by confidence, descending
    keep = []

    while len(order) > 0:
        current = order[0]
        keep.append(current)
        remaining = order[1:]

        survivors = [
            idx for idx in remaining
            if iou(boxes[current, :4], boxes[idx, :4]) <= iou_threshold
        ]
        order = np.array(survivors, dtype=int)

    return boxes[keep]


kept_boxes = non_max_suppression(raw_boxes, iou_threshold=0.4)
print(f"{len(kept_boxes)} boxes survive NMS (down from {len(raw_boxes)} raw detections):\n")
print(kept_boxes)


3 boxes survive NMS (down from 6 raw detections):

[[100.    40.   112.    54.     0.91]
 [200.    60.   216.    76.     0.83]
 [300.    45.   309.    56.     0.31]]


## Interpreting the result

NMS should collapse the pipeline's 6 raw, overlapping detections down to **3** —
one per genuinely distinct region: candidate A (the strongest of its 3 duplicates),
candidate B (the strongest of its 2 duplicates), and candidate C (the lone weak detection,
kept because nothing else overlaps it). This mirrors exactly what Chapter 02 describes: a
raw YOLO pass over-produces boxes per true object, and NMS — not the network itself — is
responsible for reducing that down to one box per object before the result is handed to
Stage 3's CNN classifier (Notebook 03) for the true/false-positive call.

Note that NMS only removes *duplicates* — it does **not** decide whether candidate C is a
true superscript or a false positive at all. That precision-oriented decision is deliberately
left to Stage 3, per the recall-first/precision-second design discussed in
`04-building-the-sequential-pipeline.md`.


In [4]:
assert len(kept_boxes) == 3, "expected NMS to collapse 6 raw boxes down to 3 distinct objects"
print("OK -- NMS collapsed 6 raw boxes down to 3 distinct candidate regions, as expected.")


OK -- NMS collapsed 6 raw boxes down to 3 distinct candidate regions, as expected.
